# Semantic Role Labeling — Rule-based vs. CRF vs. BERT Fine-tuning

**Project:** Cross-lingual (English/Spanish) Semantic Role Labeling comparison
**Datasets:** PropBank-EWT (English), PropBank-ES / Universal Propositions on AnCora (Spanish)
**Models:** Rule-based (spaCy dependency parsing), CRF (sklearn-crfsuite), BERT fine-tuning (Hugging Face Transformers)
**Goal:** Benchmark all three paradigms under identical preprocessing/evaluation conditions and quantify the cross-lingual performance gap.

> Notebook runs on Google Colab with a T4 GPU. Update the Google Drive paths (`PROPBANK_PATH`, `PROPBANK_ES_PATH`, `MODEL_SAVE_DIR`) to point to your own copy of the preprocessed data.

In [ ]:
# GPU'da olduğundan emin ol (Runtime > Change runtime type > T4 GPU)
!pip install transformers datasets seqeval evaluate sklearn-crfsuite spacy -q
!python -m spacy download en_core_web_trf -q
!python -m spacy download es_dep_news_trf -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 3.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 MB 3.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_dep_news_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import json, random, time, csv, os
from collections import defaultdict
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import spacy
from spacy.tokens import Doc
import sklearn_crfsuite

from google.colab import drive
drive.mount('/content/drive')

# Dosya yollarını kendi Drive'ına göre ayarla
PROPBANK_PATH = '/content/drive/MyDrive/colab_upload/SRL_data/propbank_preprocessed.json'
PROPBANK_ES_PATH = '/content/drive/MyDrive/colab_upload/SRL_data/propbank_es_preprocessed.json'
MODEL_SAVE_DIR = '/content/drive/MyDrive/colab_upload/SRL_data/models'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Gelişmiş TRF modellerini yüklüyoruz
nlp_en = spacy.load("en_core_web_trf", disable=['ner', 'lemmatizer'])
nlp_es = spacy.load("es_dep_news_trf", disable=['ner', 'lemmatizer'])

# --- Veri Yükleme ve Metrik Fonksiyonların (Orijinal) ---
def load_dataset(json_path):
    with open(json_path, encoding='utf-8') as f:
        data = json.load(f)
    train, dev, test = [], [], []
    for inst in data['instances']:
        sp = inst.get('split', 'train')
        (train if sp == 'train' else dev if sp == 'dev' else test).append(inst)
    return train, dev, test

def truncate(items, max_n, seed=42):
    return items if len(items) <= max_n else random.Random(seed).sample(items, max_n)

def get_label_set(*lists):
    s = set()
    for lst in lists:
        for inst in lst:
            s.update(inst['labels'])
    s.discard('O')
    return ['O'] + sorted(s)

def bio_to_spans(labels):
    spans = []; cur_role = None; cur_start = None
    for i, lbl in enumerate(labels):
        if lbl.startswith('B-'):
            if cur_role is not None: spans.append((cur_role, cur_start, i))
            cur_role, cur_start = lbl[2:], i
        elif lbl.startswith('I-'):
            role = lbl[2:]
            if cur_role != role:
                if cur_role is not None: spans.append((cur_role, cur_start, i))
                cur_role, cur_start = role, i
        else:
            if cur_role is not None: spans.append((cur_role, cur_start, i))
            cur_role = cur_start = None
    if cur_role is not None: spans.append((cur_role, cur_start, len(labels)))
    return spans

def _prf(tp, fp, fn):
    p = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    r = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f = 2*p*r/(p+r) if (p+r) > 0 else 0.0
    return round(p,4), round(r,4), round(f,4)

def compute_metrics(gold_list, pred_list, skip_labels=('O','V')):
    per_role = defaultdict(lambda: {'tp':0,'fp':0,'fn':0})
    total_tp = total_fp = total_fn = 0
    for gold_seq, pred_seq in zip(gold_list, pred_list):
        n = max(len(gold_seq), len(pred_seq))
        gold_seq = gold_seq + ['O']*(n-len(gold_seq))
        pred_seq = pred_seq + ['O']*(n-len(pred_seq))
        gold_spans = {s for s in bio_to_spans(gold_seq) if s[0] not in skip_labels}
        pred_spans = {s for s in bio_to_spans(pred_seq) if s[0] not in skip_labels}
        for sp in pred_spans:
            if sp in gold_spans:
                per_role[sp[0]]['tp'] += 1; total_tp += 1
            else:
                per_role[sp[0]]['fp'] += 1; total_fp += 1
        for sp in gold_spans:
            if sp not in pred_spans:
                per_role[sp[0]]['fn'] += 1; total_fn += 1
    p, r, f = _prf(total_tp, total_fp, total_fn)
    per_role_out = {}
    for role, c in sorted(per_role.items()):
        rp, rr, rf = _prf(c['tp'], c['fp'], c['fn'])
        per_role_out[role] = {'precision':rp,'recall':rr,'f1':rf,'support':c['tp']+c['fn']}
    return {'precision':p,'recall':r,'f1':f, 'tp':total_tp,'fp':total_fp,'fn':total_fn,'per_role':per_role_out}

def print_results(name, results):
    sep = '-'*60
    print(f'\n{sep}\n  {name}\n{sep}')
    print(f'  Precision={results["precision"]:.4f}  Recall={results["recall"]:.4f}  F1={results["f1"]:.4f}')
    print(f'  TP={results["tp"]}  FP={results["fp"]}  FN={results["fn"]}')
    if results['per_role']:
        print(f'\n  {"Role":<20} {"Prec":>8} {"Rec":>8} {"F1":>8} {"Sup":>6}')
        for role, m in sorted(results['per_role'].items(), key=lambda x: -x[1]['support'])[:15]:
            print(f'  {role:<20} {m["precision"]:>8.4f} {m["recall"]:>8.4f} {m["f1"]:>8.4f} {m["support"]:>6}')
    print(sep)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda


In [ ]:
# --- Orijinal Sözlüklerin ---
_DEP_TO_ROLE_PROPBANK = {
    'nsubj':'ARG0','nsubjpass':'ARG1','csubj':'ARG0','csubjpass':'ARG1',
    'dobj':'ARG1','obj':'ARG1','iobj':'ARG2','dative':'ARG2',
    'attr':'ARG1','oprd':'ARG1','acomp':'ARG2','xcomp':'ARG1','ccomp':'ARG1','agent':'ARG0',
}
_PREP_TO_ARGM_PROPBANK = {
    'in':'LOC','at':'LOC','on':'LOC','under':'LOC','over':'LOC','near':'LOC',
    'beside':'LOC','along':'LOC','above':'LOC','below':'LOC','behind':'LOC','inside':'LOC','outside':'LOC',
    'to':'DIR','into':'DIR','onto':'DIR','toward':'DIR','towards':'DIR','from':'DIR',
    'after':'TMP','before':'TMP','during':'TMP','since':'TMP','until':'TMP','within':'TMP',
    'by':'MNR','with':'MNR','without':'MNR','via':'MNR','through':'MNR','as':'MNR',
    'because':'CAU','due':'CAU','for':'PNC','of':'ADV','about':'ADV',
}
def _ancora_role(r): return ('A' + r[3:]) if r.startswith('ARG') else r
_DEP_TO_ROLE_PROPBANK_ES = {k: _ancora_role(v) for k, v in _DEP_TO_ROLE_PROPBANK.items()}
_PREP_TO_ARGM_PROPBANK_ES = {
    'en':'LOC','sobre':'LOC','bajo':'LOC','cerca':'LOC','junto':'LOC','dentro':'LOC','fuera':'LOC',
    'a':'DIR','hacia':'DIR','desde':'DIR',
    'hasta':'TMP','antes':'TMP','despues':'TMP','durante':'TMP',
    'por':'MNR','con':'MNR','sin':'MNR',
    'porque':'CAU','para':'PNC','de':'ADV',
}

class RuleBasedSRL:
    def __init__(self, mode='propbank', batch_size=256):
        # İYİLEŞTİRME 1: TRF modelleri kullanılarak parse kalitesi artırıldı
        self.nlp = nlp_es if mode == 'propbank_es' else nlp_en
        self.mode = mode
        self.batch_size = batch_size

        if mode == 'propbank_es':
            self._dep_map  = _DEP_TO_ROLE_PROPBANK_ES
            self._prep_map = _PREP_TO_ARGM_PROPBANK_ES
            self._mnr = 'AM-MNR'; self._neg = 'AM-NEG'; self._dis = 'AM-DIS'
        else:
            self._dep_map  = _DEP_TO_ROLE_PROPBANK
            self._prep_map = _PREP_TO_ARGM_PROPBANK
            self._mnr = 'ARGM-MNR'; self._neg = 'ARGM-NEG'; self._dis = 'ARGM-DIS'

    def predict_batch(self, instances):
        all_words = [inst['words'] for inst in instances]
        pred_idxs = [inst['predicate_idx'] for inst in instances]
        all_docs = []
        for st in range(0, len(all_words), self.batch_size):
            batch = [Doc(self.nlp.vocab, words=w) for w in all_words[st:st+self.batch_size]]
            for _, comp in self.nlp.pipeline:
                batch = list(comp.pipe(batch))
            all_docs.extend(batch)
        return [self._predict_from_doc(doc, p, len(w)) for doc, p, w in zip(all_docs, pred_idxs, all_words)]

    def _role_for(self, tok):
        dep = tok.dep_
        if dep == 'agent':                  return self._dep_map.get('agent')
        role = self._dep_map.get(dep)
        if role is not None:                return role
        if dep == 'prep' or dep == 'case':  return self._prep_map.get(tok.lower_)
        if dep == 'advmod':                 return self._mnr
        if dep == 'neg' and self._neg:      return self._neg
        if dep == 'mark' and self._dis:     return self._dis
        if dep == 'npadvmod':               return self._mnr
        return None

    def _predict_from_doc(self, doc, predicate_idx, n):
        labels = ['O']*n
        if predicate_idx < 0 or predicate_idx >= n or predicate_idx >= len(doc): return labels
        labels[predicate_idx] = 'B-V'
        pred_tok = doc[predicate_idx]
        for tok in pred_tok.children:
            if tok.i == predicate_idx or tok.i >= n: continue
            role = self._role_for(tok)
            if role is None: continue
            span_idxs = sorted(s.i for s in tok.subtree if 0 <= s.i < n and s.i != predicate_idx)
            if not span_idxs: continue
            if span_idxs[-1] - span_idxs[0] + 1 != len(span_idxs):
                if labels[tok.i] == 'O': labels[tok.i] = f'B-{role}'
                continue
            start = span_idxs[0]
            if labels[start] != 'O': continue
            labels[start] = f'B-{role}'
            for idx in span_idxs[1:]:
                if labels[idx] == 'O': labels[idx] = f'I-{role}'
        return labels

In [ ]:
def _word_shape(word):
    out = []; prev = None
    for ch in word:
        c = 'X' if ch.isupper() else 'x' if ch.islower() else 'D' if ch.isdigit() else ch
        if c != prev: out.append(c); prev = c
    return ''.join(out)[:8]

def _safe(seq, i, default=''):
    return seq[i] if 0 <= i < len(seq) else default

# İYİLEŞTİRME 2: Path to predicate fonksiyonu
def get_path_to_predicate(token_idx, pred_idx, doc):
    if pred_idx >= len(doc) or token_idx >= len(doc): return "none"
    token = doc[token_idx]
    pred_token = doc[pred_idx]
    path = []
    head = token.head
    while head != pred_token and head != head.head:
        path.append(head.dep_)
        head = head.head
    return "-".join(path) if path else "none"

def word2features(words, pos_tags, token_idx, pred_idx, dep_rels=None, head_idxs=None, doc=None):
    word = words[token_idx]
    pos  = pos_tags[token_idx] if token_idx < len(pos_tags) else 'NN'
    dist = token_idx - pred_idx
    abs_d = abs(dist)

    feats = {
        'bias':1.0,
        'lower':word.lower(),'shape':_word_shape(word),
        'pos':pos,'pos2':pos[:2],
        'prefix2':word[:2].lower(),'prefix3':word[:3].lower(),
        'suffix2':word[-2:].lower(),'suffix3':word[-3:].lower(),
        'is_upper':word.isupper(),'is_title':word.istitle(),
        'is_digit':word.isdigit(),'is_alpha':word.isalpha(),
        'is_punct':not word.isalnum(),'is_pred':token_idx==pred_idx,
        'abs_dist':abs_d,'dist':dist,
        'dist_bin':('close' if abs_d<=2 else 'mid' if abs_d<=5 else 'far' if abs_d<=10 else 'very_far'),
        'dist_sign':('before' if dist<0 else 'after' if dist>0 else 'eq'),
    }

    # İYİLEŞTİRME 3: Path ve Voice feature'larının orijinal özelliklere eklenmesi
    if doc and pred_idx < len(doc):
        feats['path_to_predicate'] = get_path_to_predicate(token_idx, pred_idx, doc)
        feats['is_passive'] = any(c.dep_ == 'auxpass' for c in doc[pred_idx].children)
        feats['dep'] = doc[token_idx].dep_

    pred_word = _safe(words, pred_idx).lower()
    pred_pos  = _safe(pos_tags, pred_idx)
    feats['pred_lower']=pred_word
    feats['pred_pos']=pred_pos
    feats['pred_pos2']=pred_pos[:2]
    feats['pred+lower']=f'{pred_word}|{word.lower()}'
    feats['pred+pos']=f'{pred_word}|{pos}'
    for off in (-2,-1,1,2):
        j = token_idx+off
        if 0 <= j < len(words):
            feats[f'w{off:+d}_lower']=words[j].lower()
            feats[f'w{off:+d}_pos']=_safe(pos_tags, j)
            feats[f'w{off:+d}_pos2']=_safe(pos_tags, j)[:2]
            feats[f'w{off:+d}_suffix3']=words[j][-3:].lower()
        else:
            feats[f'w{off:+d}_BOS_EOS']=True
    if token_idx == 0: feats['BOS']=True
    if token_idx == len(words)-1: feats['EOS']=True
    return feats

# İYİLEŞTİRME 4: NLTK yerine spaCy kullanarak İngilizce tagger'ın İspanyolca'yı bozmasını engelleme
def batch_sent2features(instances, nlp_model):
    all_words = [inst['words'] for inst in instances]
    pred_idxs = [inst['predicate_idx'] for inst in instances]

    print(f'    spaCy Parsing & POS tagging ({len(instances):,} cümle)...', flush=True)
    all_docs = []
    # Batch processing with spacy pipe for speed
    docs = nlp_model.pipe([" ".join(w) for w in all_words], batch_size=256)
    all_docs = list(docs)

    # spaCy tag'lerini kullanıyoruz
    all_pos = [[token.pos_ for token in doc] for doc in all_docs]

    return [
        [word2features(w, p, i, pi, doc=doc) for i in range(len(w))]
        for w, p, pi, doc in zip(all_words, all_pos, pred_idxs, all_docs)
    ]

class CRFModel:
    # İYİLEŞTİRME 5: max_iterations 100'den 300'e çıkarıldı (Tam yakınsama için)
    def __init__(self, max_iterations=300, c1=0.1, c2=0.1):
        self.model = sklearn_crfsuite.CRF(
            algorithm='lbfgs', c1=c1, c2=c2,
            max_iterations=max_iterations,
            all_possible_transitions=True,
            verbose=True,
        )
        self._fitted = False

    def fit(self, train_instances, language):
        nlp_model = nlp_es if language == "propbank_es" else nlp_en
        print(f'  CRF: özellik çıkarımı (train, {len(train_instances):,} cümle)...')
        X = batch_sent2features(train_instances, nlp_model)
        y = [inst['labels'] for inst in train_instances]
        print(f'  CRF: eğitim başlıyor (lbfgs, max_iter={self.model.max_iterations})...', flush=True)
        self.model.fit(X, y)
        self._fitted = True
        print('  CRF: eğitim tamamlandı.')

    def predict(self, instances, language):
        assert self._fitted
        nlp_model = nlp_es if language == "propbank_es" else nlp_en
        print(f'  CRF: özellik çıkarımı (predict, {len(instances):,} cümle)...')
        X = batch_sent2features(instances, nlp_model)
        return self.model.predict(X)

In [ ]:
PRED_START_TOKEN = '[unused0]'
PRED_END_TOKEN   = '[unused1]'
IGNORE_IDX       = -100

class SRLDataset(Dataset):
    def __init__(self, instances, tokenizer, label2id, max_length=128):
        self.instances  = instances
        self.tokenizer  = tokenizer
        self.label2id   = label2id
        self.max_length = max_length

    def _encode(self, inst):
        words, pred_idx, gold_labels = inst['words'], inst['predicate_idx'], inst['labels']
        marked_words, marked_labels = [], []
        for i, (w, lbl) in enumerate(zip(words, gold_labels)):
            if i == pred_idx:
                marked_words += [PRED_START_TOKEN, w, PRED_END_TOKEN]
                marked_labels += [None, lbl, None]
            else:
                marked_words.append(w)
                marked_labels.append(lbl)

        enc = self.tokenizer(
            marked_words, is_split_into_words=True,
            max_length=self.max_length, padding='max_length',
            truncation=True, return_tensors='pt',
        )
        word_ids = enc.word_ids()
        aligned, prev_wid = [], None
        for wid in word_ids:
            if wid is None: aligned.append(IGNORE_IDX)
            elif wid != prev_wid:
                lbl = marked_labels[wid]
                aligned.append(self.label2id.get(lbl, IGNORE_IDX) if lbl is not None else IGNORE_IDX)
            else: aligned.append(IGNORE_IDX)
            prev_wid = wid

        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(aligned, dtype=torch.long),
        }

    def __len__(self): return len(self.instances)
    def __getitem__(self, idx): return self._encode(self.instances[idx])

class BertSRLModel:
    # İYİLEŞTİRME 6: Default epoch sayısı 3'ten 5'e çıkarıldı. İspanyolca için mBERT yerine BETO.
    def __init__(self, label_set, language="propbank", max_length=128, batch_size=32, epochs=5, lr=2e-5):
        self.label_set  = label_set
        self.label2id   = {l: i for i, l in enumerate(label_set)}
        self.id2label   = {i: l for i, l in enumerate(label_set)}
        self.max_length = max_length
        self.batch_size = batch_size
        self.epochs     = epochs
        self.lr         = lr
        self.device     = device

        model_name = "dccuchile/bert-base-spanish-wwm-cased" if language == "propbank_es" else "bert-base-cased"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.add_special_tokens({'additional_special_tokens': [PRED_START_TOKEN, PRED_END_TOKEN]})
        self.model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(label_set), id2label=self.id2label,
            label2id=self.label2id, ignore_mismatched_sizes=True,
        )
        self.model.resize_token_embeddings(len(self.tokenizer))
        self.model.to(self.device)
        self._fitted = False

    def fit(self, train_instances, dev_instances=None):
        print(f'Fine-tuning on {len(train_instances):,} instances...')
        train_ds = SRLDataset(train_instances, self.tokenizer, self.label2id, self.max_length)
        train_dl = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True, num_workers=2, pin_memory=True)

        optimizer = AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        total_steps = len(train_dl) * self.epochs
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            total_loss = 0.0
            bar = tqdm(train_dl, desc=f'Epoch {epoch}/{self.epochs}')
            for batch in bar:
                batch = {k: v.to(self.device) for k, v in batch.items()}
                loss = self.model(**batch).loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                total_loss += loss.item()
                bar.set_postfix(loss=f'{loss.item():.3f}')
            print(f'  Epoch {epoch} avg loss: {total_loss/len(train_dl):.4f}')
        self._fitted = True

    def predict(self, instances):
        assert self._fitted
        ds = SRLDataset(instances, self.tokenizer, self.label2id, self.max_length)
        dl = DataLoader(ds, batch_size=self.batch_size, shuffle=False, num_workers=2, pin_memory=True)
        self.model.eval()
        all_preds = []
        with torch.no_grad():
            for batch in tqdm(dl, desc='Predicting'):
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                label_ids      = batch['labels'].numpy()
                logits = self.model(input_ids=input_ids, attention_mask=attention_mask).logits
                pred_ids = logits.argmax(dim=-1).cpu().numpy()
                for pred_row, lbl_row in zip(pred_ids, label_ids):
                    seq = [self.id2label[p] for p, l in zip(pred_row, lbl_row) if l != IGNORE_IDX]
                    all_preds.append(seq)

        aligned = []
        for inst, pred_seq in zip(instances, all_preds):
            n = len(inst['words'])
            aligned.append(pred_seq[:n] if len(pred_seq) >= n else pred_seq + ['O'] * (n - len(pred_seq)))
        return aligned

In [ ]:
def run_all_experiments(dataset_name, data_path, max_instances=None):
    print(f'\n{"="*60}\n  DATASET: {dataset_name.upper()}')
    train, dev, test = load_dataset(data_path)

    if max_instances:
        train = truncate(train, max_instances)
        test  = truncate(test, max(max_instances//5, 50))

    print(f'  Train={len(train):,}  Dev={len(dev):,}  Test={len(test):,}\n{"="*60}')
    gold = [inst['labels'] for inst in test]
    label_set = get_label_set(train, dev, test)
    summary = {}

    # 1. Kural Tabanlı (TRF Modeli ile)
    print('\n[1/3] Rule-based SRL')
    t0 = time.time()
    rb = RuleBasedSRL(mode=dataset_name)
    rb_preds = rb.predict_batch(test)
    rb_t = time.time() - t0
    rb_res = compute_metrics(gold, rb_preds)
    print_results(f'Rule-based [{dataset_name}]', rb_res)
    summary['rule_based'] = {**rb_res, 'time_s': round(rb_t, 2)}

    # 2. CRF (Gelişmiş Özellikler ve Max_Iter=300 ile)
    print('\n[2/3] CRF SRL')
    t0 = time.time()
    crf = CRFModel(max_iterations=300)
    crf.fit(train, language=dataset_name)
    crf_preds = crf.predict(test, language=dataset_name)
    crf_t = time.time() - t0
    crf_res = compute_metrics(gold, crf_preds)
    print_results(f'CRF [{dataset_name}]', crf_res)
    summary['crf'] = {**crf_res, 'time_s': round(crf_t, 2)}

    # 3. BERT (5 Epoch, BETO entegre)
    print('\n[3/3] BERT SRL')
    t0 = time.time()
    bert_model = BertSRLModel(label_set=label_set, language=dataset_name, batch_size=32, epochs=5)
    bert_model.fit(train, dev_instances=dev)
    bert_preds = bert_model.predict(test)
    bert_t = time.time() - t0
    bert_res = compute_metrics(gold, bert_preds)
    print_results(f'BERT [{dataset_name}]', bert_res)
    summary['bert'] = {**bert_res, 'time_s': round(bert_t, 2)}

    return summary

# MAX = 5000 ile test edebilirsin, full eğitim için MAX = None yap
MAX = None
all_results = {}

all_results['propbank'] = run_all_experiments('propbank', PROPBANK_PATH, MAX)
all_results['propbank_es'] = run_all_experiments('propbank_es', PROPBANK_ES_PATH, MAX)

# --- CSV Kaydetme ---
print('\n' + '='*72)
print('  TÜM SONUÇLAR KARŞILAŞTIRMA TABLOSU')
print('='*72)
print(f'  {"Dataset":<12} {"Method":<12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Time(s)":>9}')
print('-'*72)
rows = []
for ds, methods in all_results.items():
    for method, res in methods.items():
        print(f'  {ds:<12} {method:<12} {res["precision"]:>10.4f} {res["recall"]:>10.4f} {res["f1"]:>10.4f} {res.get("time_s",0):>9.1f}')
        rows.append({'dataset':ds,'method':method,'precision':res['precision'],'recall':res['recall'],'f1':res['f1'],'time_s':res.get('time_s',0)})
print('='*72)

RESULTS_PATH = os.path.join(MODEL_SAVE_DIR, 'results_all_optimized.csv')
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
with open(RESULTS_PATH, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print(f'\nCSV kaydedildi: {RESULTS_PATH}')


  DATASET: PROPBANK
  Train=40,424  Dev=4,963  Test=4,795

[1/3] Rule-based SRL

------------------------------------------------------------
  Rule-based [propbank]
------------------------------------------------------------
  Precision=0.4288  Recall=0.3222  F1=0.3679
  TP=3008  FP=4007  FN=6329

  Role                     Prec      Rec       F1    Sup
  ARG1                   0.6505   0.4350   0.5214   3170
  ARG0                   0.5539   0.6460   0.5964   1726
  ARG2                   0.8235   0.2738   0.4109   1125
  ARGM-TMP               0.0000   0.0000   0.0000    542
  ARGM-ADV               0.0000   0.0000   0.0000    493
  ARGM-MOD               0.0000   0.0000   0.0000    441
  ARGM-ADJ               0.0000   0.0000   0.0000    226
  ARGM-NEG               0.9101   0.7500   0.8223    216
  ARGM-LOC               0.0000   0.0000   0.0000    206
  ARGM-DIS               0.0000   0.0000   0.0000    181
  ARGM-MNR               0.0496   0.2973   0.0850    148
  ARGM-EXT    

loading training data to CRFsuite: 100%|██████████| 40424/40424 [00:36<00:00, 1098.34it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 1297766
Seconds required: 11.211

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 300
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=83.46 loss=2874959.44 active=1287680 feature_norm=1.00
Iter 2   time=41.81 loss=2612110.83 active=1259211 feature_norm=0.91
Iter 3   time=167.38 loss=2155156.64 active=1255295 feature_norm=0.84
Iter 4   time=124.33 loss=2146101.96 active=1283623 feature_norm=0.87
Iter 5   time=41.12 loss=2047837.86 active=1285189 feature_norm=0.90
Iter 6   time=40.98 loss=1957225.71 active=1279568 feature_norm=0.99
Iter 7   time=123.22 loss=1872035.38 active=1221615 feature_norm=1.13
Iter 8   time=41.21 loss=1722352.27 active=1200242 feature_norm=1.49
Iter 9   time=41.28 loss=1656681.06 activ